# 1. Import Libraries

In [8]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import pipeline

# 2. Get Model

In [15]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6919.15it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


# 3. Test Prompt

In [20]:
prompt = """
Paraphrase the following sentence without changing its positive sentiment:
"Film ini sangat menarik dan akting para pemainnya luar biasa."
"""

def prompt_llm(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt_llm(prompt)

'"The film is a great film to watch and watch for the first time."'

# 4. Count Dataset Sentiments

In [38]:
import pandas as pd

df = pd.read_csv("../../dataset/dataset.csv")

sentiment_counts = {}

for index, row in df.iterrows():
    sentiment = row["manual sentiment"]
    
    if (sentiment not in sentiment_counts):
        sentiment_counts[sentiment] = 1
    else:
        sentiment_counts[sentiment] += 1

highest_sentiment_tuple = ("", 0)

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    if (value > highest_sentiment_tuple[1]):
        highest_sentiment_tuple = (key, value)

sentiment_generation_count = {}

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    _, highest_sentiment_count = highest_sentiment_tuple

    sentiment_generation_count[key] = highest_sentiment_count -  value

print("The LLM needs to generate more data according to these counts: ")
print(sentiment_generation_count)

The LLM needs to generate more data according to these counts: 
{'Positive': 348, 'Negative': 0, 'Neutral': 280}


# 5. Generate Prompts

In [40]:
def create_prompt(sentiment: str) -> str:
    prompt_template = f"""
    You are an Indonesian news writer.

    Write a short Indonesian news article about Donald Trump's tariff policies
    and their impact on Indonesia.

    Requirements:
    - The article must be written in formal Indonesian language.
    - The article should sound realistic and journalistic.
    - Keep the article concise (3–5 sentences).
    - The overall sentiment of the article must be {sentiment}.
    - Do not explicitly mention the sentiment label.
    - Focus on economic, trade, export, import, or diplomatic impacts.
    - Use varied wording and natural phrasing.

    Generate only the news article.
    """

    return prompt_template

prompts_per_sentiment = {}

for key in sentiment_counts.keys():
    prompts_per_sentiment[key] = create_prompt(key)

prompts_per_sentiment

{'Positive': "\n    You are an Indonesian news writer.\n\n    Write a short Indonesian news article about Donald Trump's tariff policies\n    and their impact on Indonesia.\n\n    Requirements:\n    - The article must be written in formal Indonesian language.\n    - The article should sound realistic and journalistic.\n    - Keep the article concise (3–5 sentences).\n    - The overall sentiment of the article must be Positive.\n    - Do not explicitly mention the sentiment label.\n    - Focus on economic, trade, export, import, or diplomatic impacts.\n    - Use varied wording and natural phrasing.\n\n    Generate only the news article.\n    ",
 'Negative': "\n    You are an Indonesian news writer.\n\n    Write a short Indonesian news article about Donald Trump's tariff policies\n    and their impact on Indonesia.\n\n    Requirements:\n    - The article must be written in formal Indonesian language.\n    - The article should sound realistic and journalistic.\n    - Keep the article conc